In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [2]:
cta_df = pd.read_parquet('../feature_engineer/output/cta_ridership_with_features.parquet')
cta_df = cta_df.reset_index(drop=True)

In [3]:
cta_df.head(10)

,station_id,stationname,date,daytype,rides,map_id,red,blue,g,brn,p,y,pnk,o,line,year,month,day,day_of_week_num,day_of_week_name,rides_quartile
0,40350,UIC-Halsted,2001-01-01,U,273,40350,False,True,False,False,False,False,False,False,blue,2001,1,1,0,Monday,low
1,41130,Halsted-Orange,2001-01-01,U,306,41130,False,False,False,False,False,False,False,True,orange,2001,1,1,0,Monday,low
2,40760,Granville,2001-01-01,U,1059,40760,True,False,False,False,False,False,False,False,red,2001,1,1,0,Monday,low
3,40070,Jackson/Dearborn,2001-01-01,U,649,40070,False,True,False,False,False,False,False,False,blue,2001,1,1,0,Monday,low
4,40090,Damen-Brown,2001-01-01,U,411,40090,False,False,False,True,False,False,False,False,brown,2001,1,1,0,Monday,low
5,40590,Damen/Milwaukee,2001-01-01,U,870,40590,False,True,False,False,False,False,False,False,blue,2001,1,1,0,Monday,low
6,40720,East 63rd-Cottage Grove,2001-01-01,U,391,40720,False,False,True,False,False,False,False,False,green,2001,1,1,0,Monday,low
7,41260,Austin-Lake,2001-01-01,U,399,41260,False,False,True,False,False,False,False,False,green,2001,1,1,0,Monday,low
8,40230,Cumberland,2001-01-01,U,788,40230,False,True,False,False,False,False,False,False,blue,2001,1,1,0,Monday,low
9,41120,35-Bronzeville-IIT,2001-01-01,U,448,41120,False,False,True,False,False,False,False,False,green,2001,1,1,0,Monday,low


# Pre-processing

## Encode categorical features

In [4]:
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
onehot_encoder = OneHotEncoder()

preprocessor = make_column_transformer(
    # (ordinal_encoder, make_column_selector(dtype_include=object)),
    (onehot_encoder, make_column_selector(dtype_include=object)),
    remainder='passthrough'
)

# Train-test split

In [10]:
X = preprocessor.fit_transform(cta_df[['line', 'year', 'month', 'day', 'day_of_week_num', 'day_of_week_name']])
y = cta_df['rides']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Models

## RF

In [11]:
clf_rf = RandomForestRegressor(n_estimators=10, random_state=42)
clf_rf.fit(X_train, y_train)
y_pred_rf = clf_rf.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_rf))

R squared:  0.23800105587659282


In [12]:
y_test

458792      1546
630508      3033
557916       177
52108       3949
930488      3039
           ...  
1055995     1284
953981      1862
608030      2273
969042     22191
597371      8321
Name: rides, Length: 419262, dtype: int64